# 07 · Sliding-window attention and the rolling KV cache

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilesh-salpe/visual-ai/blob/main/build-it/architecture/07_sliding_window.ipynb) &nbsp; 📖 Concept deck: [Long context II: sliding window and ALiBi](https://nilesh-salpe.github.io/visual-ai/architecture/08-sliding-window/) &nbsp; ⏱️ Suggested time: **40 min**

**You will build:** Mistral 7B's sliding-window mask, a rolling-buffer KV cache whose memory never grows, and a test that decoding with the buffer gives exactly the same outputs as full masked attention.

**How to use this notebook**
1. Run the **Setup** cell.
2. For each task, fill in the `TODO` in the code cell, then run the **Tests** cell under it. You'll see ⏳ (not done yet), ✅ (passed) or ❌ (failed, with the reason).
3. Stuck? The **Reference solutions** at the bottom are collapsed. Try first.

Everything runs on CPU with NumPy, so no GPU is needed.

In [ ]:
# @title Setup: run this first
import numpy as np, math, time, itertools, collections
np.set_printoptions(precision=4, suppress=True)

def check(name, test, impl):
    """Run `test(impl)` and report: ⏳ not implemented, ✅ passed, ❌ failed with the reason."""
    try:
        test(impl)
        print(f"✅ {name}: all tests passed")
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet. Fill in the TODO in the cell above, then re-run this cell.")
    except AssertionError as e:
        print(f"❌ {name}: {e}")


def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def _swa_ref(q, k, v, W):
    T = len(q); i, j = np.indices((T, T))
    s = q @ k.T / np.sqrt(q.shape[1])
    return softmax(np.where((j <= i) & (i - j < W), s, -np.inf)) @ v

## Task 1: The sliding-window mask

Each query $i$ attends to itself and the previous $W-1$ tokens: allowed iff $j \le i$ and $i - j < W$.
(Mistral: position $i$ attends to positions between $i-W$ and $i$ of the previous layer. Off-by-one conventions vary between codebases. Here a window holds exactly $W$ keys, matching a cache of size $W$.)

In [ ]:
def sliding_window_mask(T, W):
    # TODO: boolean [T, T]
    raise NotImplementedError

In [ ]:
# @title Tests for Task 1: run after implementing
def _test_sliding_window_mask(f):
    M = f(6, 3)
    assert M.dtype == bool and M.shape == (6, 6)
    assert M[5, 5] and M[5, 3] and not M[5, 2], "row 5 with W=3 sees columns 3, 4, 5"
    assert not M[2, 3], "still causal"
    assert np.all(M.sum(1) == np.minimum(np.arange(1, 7), 3)), "each row sees min(i+1, W) keys"

check("sliding_window_mask", _test_sliding_window_mask, sliding_window_mask)

## Task 2: A rolling-buffer KV cache

Mistral 7B: *"The cache has a fixed size of W, and the keys and values for the timestep i are stored in position i mod W of the cache."* Old entries are simply overwritten.
Implement `append(k, v)` and `get()`. `get()` returns the stored keys and values **in chronological order** (oldest first) as two arrays of shape [min(t, W), d].

In [ ]:
class RollingKVCache:
    def __init__(self, W, d):
        self.W, self.t = W, 0
        self.K = np.zeros((W, d)); self.V = np.zeros((W, d))

    def append(self, k, v):
        # TODO: write at slot t % W, then t += 1
        raise NotImplementedError

    def get(self):
        # TODO: return (K, V) of the last min(t, W) steps, oldest first
        raise NotImplementedError

In [ ]:
# @title Tests for Task 2: run after implementing
def _test_RollingKVCache(cls):
    c = cls(4, 2)
    for t in range(10):
        c.append(np.array([t, t]), np.array([-t, -t]))
        K, V = c.get()
        want = np.arange(max(0, t - 3), t + 1)
        assert K.shape == (len(want), 2), f"after {t + 1} appends expected {len(want)} entries, got {K.shape[0]}"
        assert np.array_equal(K[:, 0], want), f"after {t + 1} appends keys should be {want.tolist()}, got {K[:, 0].tolist()}"
        assert np.array_equal(V[:, 0], -want), "values out of order"
    assert c.K.shape == (4, 2), "the buffer itself must never grow"

check("RollingKVCache", _test_RollingKVCache, RollingKVCache)

## Task 3: Decode with the rolling cache

Process tokens one at a time: append $(k_t, v_t)$ to a `RollingKVCache`, then attend $q_t$ over whatever `get()` returns.
The test checks that the result equals `softmax(masked scores) @ v` with the sliding-window mask, **exactly**. The cache needs $W$ slots no matter how long the sequence gets.

In [ ]:
def decode_with_rolling_cache(q, k, v, W, cache_cls=None):
    """q, k, v: [T, d]. Returns [T, d], one token at a time."""
    cache_cls = cache_cls or RollingKVCache
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 3: run after implementing
def _test_decode_with_rolling_cache(f):
    r = np.random.default_rng(1); q, k, v = r.normal(size=(3, 20, 8))
    for W in [1, 4, 7, 20, 32]:
        out = f(q, k, v, W)
        assert out.shape == (20, 8)
        assert np.allclose(out, _swa_ref(q, k, v, W), atol=1e-10), f"W={W}: rolling-cache decode differs from masked sliding-window attention"

check("decode_with_rolling_cache", _test_decode_with_rolling_cache, decode_with_rolling_cache)

---
## Reference solutions
Try the tasks first. Each cell below is collapsed in Colab: double-click it (or click "Show code") to reveal the solution. Running it also checks the reference against the same tests.

In [ ]:
# @title Solution for Task 1: The sliding-window mask
def sliding_window_mask_solution(T, W):
    i, j = np.indices((T, T))
    return (j <= i) & (i - j < W)

check("sliding_window_mask (reference)", _test_sliding_window_mask, sliding_window_mask_solution)

In [ ]:
# @title Solution for Task 2: A rolling-buffer KV cache
class RollingKVCache_solution:
    def __init__(self, W, d):
        self.W, self.t = W, 0
        self.K = np.zeros((W, d)); self.V = np.zeros((W, d))

    def append(self, k, v):
        self.K[self.t % self.W] = k; self.V[self.t % self.W] = v
        self.t += 1

    def get(self):
        n = min(self.t, self.W)
        idx = [(self.t - n + i) % self.W for i in range(n)]
        return self.K[idx], self.V[idx]

check("RollingKVCache (reference)", _test_RollingKVCache, RollingKVCache_solution)

In [ ]:
# @title Solution for Task 3: Decode with the rolling cache
def decode_with_rolling_cache_solution(q, k, v, W, cache_cls=None):
    cache = (cache_cls or RollingKVCache_solution)(W, k.shape[1]); out = []
    for t in range(len(q)):
        cache.append(k[t], v[t]); K, V = cache.get()
        out.append(softmax(q[t] @ K.T / np.sqrt(q.shape[1])) @ V)
    return np.array(out)

check("decode_with_rolling_cache (reference)", _test_decode_with_rolling_cache, decode_with_rolling_cache_solution)

---
## Experiments
**Stacked windows see much further than one window.** Information can hop $W-1$ tokens per layer, so after $L$ layers the receptive field is about $L\cdot(W-1)+1$. Below we compute it exactly by composing masks, then apply the formula to Mistral 7B ($W=4096$, 32 layers). The paper quotes about 131K tokens ($W\times L$). Note this is a *theoretical* span: information from far away has to survive many hops.

In [ ]:
T, W = 64, 5; M = sliding_window_mask_solution(T, W).astype(int); reach = np.eye(T, dtype=int)
for layer in range(1, 9):
    reach = (M @ reach > 0).astype(int)
    print(f"layers={layer}: last token can be influenced by {reach[-1].sum():2d} tokens  (formula L*(W-1)+1 = {layer * (W - 1) + 1})")
print(f"\nMistral 7B: 32 * (4096 - 1) + 1 = {32 * 4095 + 1:,} tokens (paper: ~131K)")
kv = lambda tokens: 2 * 32 * 8 * 128 * 2 * tokens   # layers * kv_heads * head_dim * bf16 bytes
print(f"KV cache at 32K tokens: full {kv(32768) / 2**30:.2f} GiB vs rolling buffer {kv(4096) / 2**30:.2f} GiB  ({32768 // 4096}x smaller, as the paper reports)")

---
## Takeaways
* Sliding-window attention costs $O(T\cdot W)$ instead of $O(T^2)$, and the KV cache is capped at $W$ entries per layer.
* Depth stacks windows: $L$ layers reach about $L\cdot W$ tokens (131K for Mistral 7B), though only indirectly.
* The rolling buffer (`i mod W`) is what makes the memory saving real at inference: 8× less cache at 32K tokens for Mistral 7B.